# Lawgic: Contextual ToS Visualizer Exploration and Prototypign

## TODO:
- hard code a single tos document lang. use an existing one in there and follow their format (i picked amazon)
- see how we can visualize the amazon documents
  - it seems like we can take the `title` from `points` and just wrap those around a central `Amazon` node. then clicking that will open the more detailed `points.quote_text`.
  - next question: how do we draw relationship?
- for demo on friday:
  - select from list of companies
  - select from list of documents per company
  - when selecting document, show all cases related to that. (use title as the node?)
  - then let's see from there...

In [ ]:
import pandas as pd
import numpy as np
import json

In [4]:
DATA_PATH = "../datasets/tos_dr/"

#### Cases

Cases refer to individual legal cases or specific issues related to the terms of service or privacy policies of a particular online service. Each case typically focuses on a specific aspect of a service's terms, such as data collection, user rights, content ownership, or security practices.

- id, a unique id for each case (incremental).
- classification, one of those values (good, bad, neutral, blocker).
- score, values range between 0 to 100.
- title.
- description.
- topic_id, connecting the case with it's topic.
- created_at.
- updated_at.
- privacy_related, a flag indicate if it's related to privacy or not.
- docbot_regex, the regex expression used to check for specific words in the quoted text.

In [14]:
tos_cases = pd.read_csv(DATA_PATH + "cases.csv")
tos_cases.head() 

,id,classification,score,title,description,topic_id,created_at,updated_at,privacy_related,docbot_regex
0,175,good,50,You can delete your content from this service,You can ask the service to remove your content...,45,2018-01-16 15:26:09.45258,2021-02-27 14:23:56.238878,NaN,NaN
1,130,bad,50,A license is kept on user-generated content ev...,The service may still use your content and dis...,30,2018-01-16 15:26:08.38805,2021-05-06 08:26:06.196896,True,NaN
2,129,bad,60,This service tracks you on other websites,The service may use tracking technologies in t...,25,2018-01-16 15:26:08.370367,2021-02-24 22:56:35.859683,True,NaN
3,237,neutral,0,You agree to comply with the law of the servic...,"By using the service, you agree to comply with...",28,2018-07-22 09:09:08.496243,2021-05-06 08:35:07.855459,NaN,NaN
4,173,bad,25,This service retains rights to your content ev...,The service may still use your content and dis...,45,2018-01-16 15:26:09.415729,2021-02-27 15:51:22.50162,NaN,NaN


#### Topics

Topics are general categories or themes that encompass various cases. They help organize and group similar cases together based on the type of issues they address. For example, "Data Collection" could be a topic that includes cases related to how a service collects and uses user data.

- id, a unique id for each topic (incremental).
- title.
- subtitle, small description.
- description.
- created_at.
- updated_at.

In [6]:
tos_topics = pd.read_csv(DATA_PATH + "topics.csv")
tos_topics.head()

,id,title,subtitle,description,created_at,updated_at
0,27,Ownership,What happens with the content you generate on ...,The content and the data you generate on servi...,2018-01-16 15:26:04.444207,2018-01-16 15:26:04.444207
1,31,Governance,Your relationship with the service and the com...,description,2018-01-16 15:26:04.535299,2018-01-16 15:26:04.535299
2,32,Guarantee,Some services will guarantee you certain featu...,description,2018-01-16 15:26:04.551182,2018-01-16 15:26:04.551182
3,35,User Involvement in Changing Terms,Are changes proposed as a request for feedback...,"The terms are a contract between two parties, ...",2018-01-16 15:26:04.629039,2018-01-16 15:26:04.629039
4,38,Changes,Changes to the Terms,description,2018-01-16 15:26:04.682205,2018-01-16 15:26:04.682205


#### Services

Services represent specific online platforms, websites, or applications that have their own terms of service and privacy policies.

- id, a unique id for each service (incremental).
- name.
- url.
- created_at.
- updated_at.
- wikipedia, wikipedia url of the service.
- keywords.
- related, connecting the service with one of known similar services in the same field.
- slug. extracted from the name, small letters, no spaces and so on.
- is_comprehensively_reviewed, a flag indicate if it's comprehensively_reviewed or not.
- rating, overall rating for the service based on the all cases.
- status, indicate if the service is deleted or not (deleted, NaN).

In [8]:
tos_services = pd.read_csv(DATA_PATH + "services.csv")
tos_services.head()

,id,name,url,created_at,updated_at,wikipedia,keywords,related,slug,is_comprehensively_reviewed,rating,status
0,440,pebble,pebble.com,2018-05-08 12:19:44.833708,2021-03-25 03:19:30.208894,NaN,NaN,NaN,pebble,False,NaN,NaN
1,353,Yammer,yammer.com,2018-05-08 12:19:43.120991,2021-03-31 01:04:42.63077,https://en.wikipedia.org/wiki/Yammer,NaN,NaN,yammer,False,NaN,NaN
2,2894,Instapaper,instapaper.com,2021-03-29 17:44:07.407582,2021-03-30 00:01:55.169887,https://en.wikipedia.org/wiki/Instapaper,NaN,NaN,instapaper,False,NaN,NaN
3,3157,UNRIC,unric.org,2021-04-16 20:31:05.868802,2021-04-17 00:00:48.616753,NaN,NaN,NaN,unric,False,NaN,NaN
4,162,Infinity,"infinity.com,comcast.com",2018-01-16 15:26:05.072526,2021-03-25 03:19:37.306995,https://en.wikipedia.org/wiki/Infinity#Branding,NaN,NaN,infinity,False,NaN,NaN


#### Points

Points are individual statements or aspects within a case that highlight important information about a service's terms of service or privacy policy. These points can be positive (e.g., strong privacy protections) or negative (e.g., data sharing with third parties).

- id, a unique id for each point (incremental).
- rank, all values are zero.
- title, mostly it's similar to case title.
- source, url of the source.
- status, one of those values (approved, declined, pending, changes-requested, disputed, draft).
- analysis.
- created_at.
- updated_at.
- service_id, connecting the point with it's service.
- quote_text, quotted text from the source which contain information for this point.
- case_id, connecting the point with the related case.
- old_id, used for data migration.
- quote_start, index of first letter of the quotted text in the document.
- quote_end, index of last letter of the quotted text in the document.
- service_needs_rating_update, all values are False.
- document_id, connecting the point with the related document.
- annotation_ref.

In [7]:
tos_points = pd.read_csv(DATA_PATH + "points.csv")
tos_points.head()

,id,rank,title,source,status,analysis,created_at,updated_at,service_id,quote_text,case_id,old_id,point_change,quote_start,quote_end,service_needs_rating_update,document_id,annotation_ref
0,4613,0,No need to register,https://rationalwiki.org/wiki/RationalWiki:Pri...,declined,There is no need to register to view or edit c...,2018-09-08 19:35:01.268707,2021-01-30 13:14:55.558818,702,NaN,320,NaN,NaN,NaN,NaN,False,NaN,NaN
1,605,0,The terms for this service are easy to read,https://groups.google.com/d/topic/tosdr/k_RmZY...,declined,Grammarly's Privacy Policy is easy to read. It...,2018-01-16 15:26:18.500569,2018-10-21 09:44:31.745032,277,NaN,199,k_RmZYGkrww,"For me, the terms of service are still long an...",NaN,NaN,False,NaN,NaN
2,7158,0,The terms for this service are easy to read,https://hp-67.com/terms.html,approved,"The Terms of Service states ""HP-67.com terms o...",2019-06-24 20:13:27.240822,2019-06-25 20:12:34.0269,1675,NaN,199,NaN,NaN,NaN,NaN,False,NaN,NaN
3,4612,0,The service provides two factor authentificati...,https://blog.seznam.cz/2018/04/seznam-cz-ma-no...,declined,The service provides two-factor authentificati...,2018-09-08 19:10:58.633819,2021-02-06 03:50:37.204911,684,NaN,312,NaN,NaN,NaN,NaN,False,NaN,NaN
4,4961,0,The service provides information about how the...,https://www.webseodesigners.com/,declined,Generated through the annotate view,2018-11-10 11:29:57.873485,2021-02-13 00:00:46.903004,820,We use this information to make a business pro...,227,NaN,NaN,10821.0,10880.0,False,725.0,NaN


In [42]:
tos_points[tos_points['status'] == 'approved'].value_counts('status')

status
approved    23968
Name: count, dtype: int64

In [40]:
tos_points.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31706 entries, 0 to 31705
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           31706 non-null  int64  
 1   rank                         31706 non-null  int64  
 2   title                        31706 non-null  object 
 3   source                       31513 non-null  object 
 4   status                       31706 non-null  object 
 5   analysis                     31364 non-null  object 
 6   created_at                   31706 non-null  object 
 7   updated_at                   31706 non-null  object 
 8   service_id                   31706 non-null  int64  
 9   quote_text                   27720 non-null  object 
 10  case_id                      31706 non-null  int64  
 11  old_id                       2045 non-null   object 
 12  point_change                 963 non-null    object 
 13  quote_start     

#### Documents

Documents refer to the original terms of service and privacy policies of the services that are being analyzed on TOSDR. These documents are the source of information for the cases, points, and ratings provided on the platform. TOSDR links to the actual documents, so users can review the full details if they choose to.

- id, a unique id for each document (incremental).
- name, name of document like privacy policy or cookies policy, etc.
- url, url of the document.
- xpath.
- text, the actual document.
- created_at.
- updated_at.
- service_id, connecting the document with it's service.
- reviewed, a flag indicate if the document has been reviewed or not.
- status, indicate if the service is deleted or not (deleted, NaN).
- crawler_server, the server used to crawl the document

In [ ]:
tos_documents = pd.read_csv(DATA_PATH + "documents.csv")

,id,name,url,xpath,text,created_at,updated_at,service_id,reviewed,status,crawler_server
0,1149,social.privacytools.io Privacy Policy,https://social.privacytools.io/terms,/html/body/div/div[2]/div/div[1]/div,\nPrivacy <p>Please review our privacy stateme...,2019-04-06 02:39:01.465351,2020-12-14 19:29:56.243824,1472,NaN,NaN,NaN
1,3719,Privacy Policy,https://www.savvasrealize.com/privacy/corporat...,/html/body/main/div/section/div/div/div[1],Privacy Policy We recently updated our Privac...,2021-01-21 11:15:08.988939,2021-01-24 08:49:24.411559,2619,NaN,NaN,NaN
2,7,Terms of Service,https://help.habbo.com/hc/en-us/articles/22164...,//div[@class='content-body article-body'],<p>\n<strong>Habbo.com - Terms of Service</st...,2018-07-17 10:11:40.691179,2020-12-11 14:20:59.218294,292,True,NaN,NaN
3,9,Terms of Service,https://500px.com/terms,//body,You need to enable JavaScript to run this app....,2018-07-17 10:11:40.759351,2021-03-30 21:11:14.01763,282,True,NaN,eu
4,1319,HERE Cookie Policy,https://legal.here.com/en-gb/privacy/cookies,/html/body/div[1]/div/div,HERE Cookie Policy Effective date:&nbsp;15/07...,2019-05-15 01:44:06.567469,2020-12-14 19:31:28.232051,1559,NaN,NaN,NaN


In [16]:
tos_documents.head()

,id,name,url,xpath,text,created_at,updated_at,service_id,reviewed,status,crawler_server
0,1149,social.privacytools.io Privacy Policy,https://social.privacytools.io/terms,/html/body/div/div[2]/div/div[1]/div,\nPrivacy <p>Please review our privacy stateme...,2019-04-06 02:39:01.465351,2020-12-14 19:29:56.243824,1472,NaN,NaN,NaN
1,3719,Privacy Policy,https://www.savvasrealize.com/privacy/corporat...,/html/body/main/div/section/div/div/div[1],Privacy Policy We recently updated our Privac...,2021-01-21 11:15:08.988939,2021-01-24 08:49:24.411559,2619,NaN,NaN,NaN
2,7,Terms of Service,https://help.habbo.com/hc/en-us/articles/22164...,//div[@class='content-body article-body'],<p>\n<strong>Habbo.com - Terms of Service</st...,2018-07-17 10:11:40.691179,2020-12-11 14:20:59.218294,292,True,NaN,NaN
3,9,Terms of Service,https://500px.com/terms,//body,You need to enable JavaScript to run this app....,2018-07-17 10:11:40.759351,2021-03-30 21:11:14.01763,282,True,NaN,eu
4,1319,HERE Cookie Policy,https://legal.here.com/en-gb/privacy/cookies,/html/body/div[1]/div/div,HERE Cookie Policy Effective date:&nbsp;15/07...,2019-05-15 01:44:06.567469,2020-12-14 19:31:28.232051,1559,NaN,NaN,NaN


In [18]:
tos_documents['name'].value_counts()

name
Privacy Policy                                   4335
Terms of Service                                 1342
Terms of Use                                      913
Privacy Policy                                    566
Terms and Conditions                              502
                                                 ... 
Shipping                                            1
Nutzungsbedingungen für den bitkasten               1
LIMITED WARRANTY                                    1
Viewing data supplement to the Privacy Policy       1
Protecting your privacy                             1
Name: count, Length: 3308, dtype: int64

#### Link `documents` and `services`

In [28]:
tos_documents.info()
print("NaN values: \n")
print(tos_documents.isna().sum())

print("------------------------")
print("services.csv: \n")
print(tos_services.info())
print("NaN values: \n")
print(tos_services.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16786 entries, 0 to 16785
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              16786 non-null  int64 
 1   name            16786 non-null  object
 2   url             16786 non-null  object
 3   xpath           13628 non-null  object
 4   text            12367 non-null  object
 5   created_at      16786 non-null  object
 6   updated_at      16786 non-null  object
 7   service_id      16786 non-null  int64 
 8   reviewed        227 non-null    object
 9   status          17 non-null     object
 10  crawler_server  12086 non-null  object
dtypes: int64(2), object(9)
memory usage: 1.4+ MB
NaN values: 

id                    0
name                  0
url                   0
xpath              3158
text               4419
created_at            0
updated_at            0
service_id            0
reviewed          16559
status            16769
crawler_server 

In [31]:
# Merge documents and services via service_id
merged_doc_services = pd.merge(tos_documents, tos_services[['id', 'name', 'url', 'related']], left_on='service_id', right_on='id', suffixes=('_doc', '_service'))
merged_doc_services.head()

,id_doc,name_doc,url_doc,xpath,text,created_at,updated_at,service_id,reviewed,status,crawler_server,id_service,name_service,url_service,related
0,1149,social.privacytools.io Privacy Policy,https://social.privacytools.io/terms,/html/body/div/div[2]/div/div[1]/div,\nPrivacy <p>Please review our privacy stateme...,2019-04-06 02:39:01.465351,2020-12-14 19:29:56.243824,1472,NaN,NaN,NaN,1472,PrivacyTools,privacytools.io,NaN
1,3719,Privacy Policy,https://www.savvasrealize.com/privacy/corporat...,/html/body/main/div/section/div/div/div[1],Privacy Policy We recently updated our Privac...,2021-01-21 11:15:08.988939,2021-01-24 08:49:24.411559,2619,NaN,NaN,NaN,2619,Savvas Learning Company,"savvasrealize.com,pk12ls.com,sso.rumba.pk12ls....",NaN
2,7,Terms of Service,https://help.habbo.com/hc/en-us/articles/22164...,//div[@class='content-body article-body'],<p>\n<strong>Habbo.com - Terms of Service</st...,2018-07-17 10:11:40.691179,2020-12-11 14:20:59.218294,292,True,NaN,NaN,292,Habbo,habbo.com,NaN
3,9,Terms of Service,https://500px.com/terms,//body,You need to enable JavaScript to run this app....,2018-07-17 10:11:40.759351,2021-03-30 21:11:14.01763,282,True,NaN,eu,282,500px,"500px.com,500px.me,500px.org,500px.net",flickr
4,1319,HERE Cookie Policy,https://legal.here.com/en-gb/privacy/cookies,/html/body/div[1]/div/div,HERE Cookie Policy Effective date:&nbsp;15/07...,2019-05-15 01:44:06.567469,2020-12-14 19:31:28.232051,1559,NaN,NaN,NaN,1559,HERE Technologies,"here.com,hereapi.com",NaN


In [43]:
merged_doc_services.value_counts('name_service')

name_service
Amazon                                 26
bunq                                   23
Albertsons Companies,(Subsidiaries)    23
SailPoint                              22
HERE Technologies                      21
                                       ..
Toll Brothers                           1
Tom Woods                               1
Tom's Hardware                          1
DeHashed                                1
Ecotricity                              1
Name: count, Length: 7901, dtype: int64

#### Merge `points` and `services`
Only include rows in `points` where `status == approved`

In [38]:
merged_pts_services = pd.merge(tos_points[tos_points['status'] == 'approved'], tos_services[['id', 'name', 'url', 'related']], left_on='service_id', right_on='id', suffixes=('_point', '_service'))
merged_pts_services.head()

,id_point,rank,title,source,status,analysis,created_at,updated_at,service_id,quote_text,...,point_change,quote_start,quote_end,service_needs_rating_update,document_id,annotation_ref,id_service,name,url,related
0,7158,0,The terms for this service are easy to read,https://hp-67.com/terms.html,approved,"The Terms of Service states ""HP-67.com terms o...",2019-06-24 20:13:27.240822,2019-06-25 20:12:34.0269,1675,NaN,...,NaN,NaN,NaN,False,NaN,NaN,1675,HP-67,hp-67.com,NaN
1,17700,0,"This service collects your IP address, which c...",https://en.help.roblox.com/hc/en-us/articles/1...,approved,"The Privacy and Cookie policy states: ""...we o...",2021-03-24 04:33:37.809282,2021-03-24 13:51:01.879363,1336,NaN,...,NaN,NaN,NaN,False,NaN,NaN,1336,Roblox,roblox.com,NaN
2,6132,0,The court of law governing the terms is in a j...,https://stackoverflow.com/legal/terms-of-servi...,approved,Generated through the annotate view,2019-03-19 12:45:34.493941,2023-07-11 06:33:59.625461,312,Both you and Stack Overflow hereby irrevocably...,...,NaN,20471.0,20749.0,False,352.0,6zrawh-0Ee66pFffOexWXQ,312,Stack Overflow,"stackoverflow.com,stackexchange.com",NaN
3,9860,0,You can request access and deletion of persona...,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:59.854515,2023-07-11 06:33:59.728921,1700,Sie haben jederzeit das Recht unentgeltlich Au...,...,NaN,5669.0,5911.0,False,1625.0,60qawB-0Ee66pG97_ve0ew,1700,Ruhrtalradweg.de,ruhrtalradweg.de,NaN
4,9859,0,Your personal data is used for limited purposes,https://www.ruhrtalradweg.de/Informationen/Dat...,approved,Generated through the annotate view,2020-07-27 13:09:25.245754,2023-07-11 06:33:59.813688,1700,"Ein Teil der Daten wird erhoben, um eine fehle...",...,NaN,5428.0,5602.0,False,1625.0,61eQ6h-0Ee66pIu8RRm9-w,1700,Ruhrtalradweg.de,ruhrtalradweg.de,NaN


In [39]:
merged_pts_services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23968 entries, 0 to 23967
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id_point                     23968 non-null  int64  
 1   rank                         23968 non-null  int64  
 2   title                        23968 non-null  object 
 3   source                       23853 non-null  object 
 4   status                       23968 non-null  object 
 5   analysis                     23872 non-null  object 
 6   created_at                   23968 non-null  object 
 7   updated_at                   23968 non-null  object 
 8   service_id                   23968 non-null  int64  
 9   quote_text                   22498 non-null  object 
 10  case_id                      23968 non-null  int64  
 11  old_id                       158 non-null    object 
 12  point_change                 340 non-null    object 
 13  quote_start     

In [44]:
amazon_docs = merged_doc_services[merged_doc_services['name_service'] == 'Amazon']
amazon_docs

,id_doc,name_doc,url_doc,xpath,text,created_at,updated_at,service_id,reviewed,status,crawler_server,id_service,name_service,url_service,related
36,40,Kindle Store Terms of Use,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Kindle Store Terms of Use <p>\n</p>\n<p>\n<st...,2018-07-17 10:11:41.892666,2018-07-17 10:11:41.892666,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
37,41,Saving Your Wi-Fi Passwords to Amazon FAQs,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Saving Your Wi-Fi Passwords to Amazon FAQs <p...,2018-07-17 10:11:41.917142,2018-07-17 10:11:41.917142,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
38,43,Amazon Drive and Prime Photos Terms of Use,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Amazon Drive and Prime Photos Terms of Use <p...,2018-07-17 10:11:41.956146,2018-07-17 10:11:41.956146,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
39,42,Amazon Device Terms of Use,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Amazon Device Terms of Use <p>\n<strong>Last ...,2018-07-17 10:11:41.935588,2020-12-11 14:53:38.826026,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
41,58,Amazon Maps Terms of Use,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Amazon Maps Terms of Use <p>\n<strong> Last U...,2018-07-17 10:11:42.278491,2020-12-11 14:56:51.961553,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
45,57,Amazon Coins Terms,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],"Amazon Coins Terms <p> Last updated: June 15,...",2018-07-17 10:11:42.257189,2020-12-11 14:56:51.966361,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
51,37,Conditions of Use,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Conditions of Use <p>\n<strong> Last updated:...,2018-07-17 10:11:41.821544,2020-12-11 14:50:21.132179,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
53,38,Amazon Privacy Notice,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Amazon.com Privacy Notice <p>\n</p>\n<p>\n<st...,2018-07-17 10:11:41.845825,2020-12-11 14:50:39.810369,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
54,39,Interest-Based Ads,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Interest-Based Ads <p>\n</p>\n<p>Interest-bas...,2018-07-17 10:11:41.868015,2020-12-11 14:53:10.678238,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
60,46,Alexa Terms of Use,https://www.amazon.com/gp/help/customer/displa...,//div[@class='help-content'],Alexa Terms of Use <p>\n<strong>Last Updated:...,2018-07-17 10:11:42.020074,2020-12-11 14:54:45.5575,190,True,NaN,NaN,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN


In [46]:
amazon_pts = merged_pts_services[merged_pts_services['name'] == 'Amazon']
amazon_pts

,id_point,rank,title,source,status,analysis,created_at,updated_at,service_id,quote_text,...,point_change,quote_start,quote_end,service_needs_rating_update,document_id,annotation_ref,id_service,name,url,related
526,1582,0,personal data is given to third parties,https://www.amazon.com/gp/help/customer/displa...,approved,Amazon may release your data when they believe...,2018-05-15 09:11:55.07478,2023-07-11 06:34:58.824486,190,This includes exchanging information with othe...,...,NaN,8238.0,8361.0,False,38.0,DoOvVB-1Ee66pF9jKV1_-g,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
533,1122,0,The service uses your personal data for advert...,https://www.amazon.com/gp/help/customer/displa...,approved,Amazon uses your personal data and your behavi...,2018-01-16 21:11:19.947881,2023-07-11 06:34:59.500821,190,"To serve you interest-based ads, we use inform...",...,NaN,525.0,643.0,False,39.0,DusOuh-1Ee66pKNnYAez1w,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
813,5925,0,This service employs separate policies for dif...,https://www.amazon.com/gp/help/customer/displa...,approved,Generated through the annotate view,2019-02-25 22:40:38.651689,2023-07-11 06:35:30.954289,190,"Please review our other policies, such as our ...",...,NaN,16304.0,16441.0,False,37.0,Iap3wB-1Ee66pMs5QtRIZA,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
816,5923,0,This service forces users into binding arbitra...,https://www.amazon.com/gp/help/customer/displa...,approved,Generated through the annotate view,2019-02-25 22:37:59.186409,2023-07-11 06:35:31.204853,190,Any dispute or claim relating in any way to yo...,...,NaN,13750.0,14060.0,False,37.0,IdDKJB-1Ee66pPftK-RB7Q,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
817,5922,0,The service is provided 'as is' and to be used...,https://www.amazon.com/gp/help/customer/displa...,approved,Generated through the annotate view,2019-02-25 22:36:35.651531,2023-07-11 06:35:31.286693,190,"THE AMAZON SERVICES AND ALL INFORMATION, CONTE...",...,NaN,12067.0,12757.0,False,37.0,Id1gfB-1Ee66pDs1B782tQ,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
830,5929,0,This service can share your personal informati...,https://www.amazon.com/gp/help/customer/displa...,approved,Generated through the annotate view,2019-02-25 23:09:59.612397,2023-07-11 06:35:32.728636,190,The Amazon Group Companies are subject to the ...,...,NaN,4616.0,4952.0,False,1051.0,IrlWrh-1Ee66pJeKgnQIfg,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
833,5926,0,Terms may be changed any time at their discre...,https://www.amazon.com/gp/help/customer/displa...,approved,Generated through the annotate view,2019-02-25 22:41:48.697187,2023-07-11 06:35:32.973948,190,We reserve the right to make changes to our si...,...,NaN,16443.0,16557.0,False,37.0,It6Z5h-1Ee66pDsGRtU7fw,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
834,5930,0,The service can delete your account without pr...,https://www.amazon.com/gp/help/customer/displa...,approved,Generated through the annotate view,2019-02-26 06:58:17.497787,2023-07-11 06:35:33.056589,190,"We may terminate the Agreement or restrict, su...",...,NaN,4363.0,4501.0,False,58.0,Iutofh-1Ee66pLPjHOSxxQ,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
2895,10790,0,Invalidity of any portion of the Terms of Serv...,https://www.amazon.com/gp/help/customer/displa...,approved,Generated through the annotate view,2020-10-27 05:04:51.367029,2023-07-11 10:05:52.384701,190,If any of these conditions shall be deemed inv...,...,NaN,16559.0,16771.0,False,37.0,hKAgQh_SEe6e1iv7gBRgHA,190,Amazon,"amazon.com,amazon.com.au,amazon.de,amazon.co.u...",NaN
4363,10791,0,This service prohibits users sending chain let...,https://www.amazon.com/gp/help/customer/displa...,approved,"Full Quote: ""You may post reviews, comments, p...",2020-10-27 05:08:27.080336,2023-07-11 10:08:32.042069,190,"commercial solicitation, chain letter

## Converting the CSVs into desired structure

Desired JSON structure for ONE company / service:
```json
{
    "service_id": 123,
    "service_name": "Amazon",
    "service_url": "amazon.com",
    "cases": [
        {
            "case_id": 1,
            "case_classification": "good",
            "case_score": 75,
            "case_title": "INSERT TITLE",
            "case_description": "INSERT DESCRIPTION",
            "case_topic": "Ownership", 
            "case_privacy_related": true,
            "points": [
                {
                    "point_id": 999,
                    "point_title": "INSERT TITLE",
                    "point_source": "INSERT URL",
                    "point_analysis": "INSERT ANALYSIS",
                    "point_quote_text": "INSERT TEXT HERE",
                    "point_quote_start": 100,
                    "point_quote_end": 950,
                    "point_document_id": 456
                }
            ]
        }
    ],
    "service_related": ["service_id_1", "service_id_2"]
}
```
NOTE: Very important that those prefixes are there so I can understand which data is coming from where.

In [ ]:
from IPython.core import display_functions
import pandas as pd
import json
import numpy as np

# Load the datasets
# We assume the datasets are located in the relative directory '../datasets/tos_dr/'
data_dir = '../datasets/tos_dr/'
df_services = pd.read_csv(data_dir + 'services.csv')
df_cases = pd.read_csv(data_dir + 'cases.csv')
df_points = pd.read_csv(data_dir + 'points.csv')
df_topics = pd.read_csv(data_dir + 'topics.csv')

# --- DATA PROCESSING RULES ---

# 1. Clean up points quote text: strip leading/trailing whitespace
df_points['quote_text'] = df_points['quote_text'].str.strip()

# 2. Map topic_id to case_topic title
# First, create a mapping from topic id to topic title
topic_map = df_topics.set_index('id')['title'].to_dict()
# Then map the topic_id in cases to create the new 'case_topic' column without outputting topic_id later
df_cases['case_topic'] = df_cases['topic_id'].map(topic_map)

# 3. Preprocess services 'related' column into an array of strings
def parse_related(related_val):
    if pd.isna(related_val) or not str(related_val).strip():
        return []
    return [x.strip() for x in str(related_val).split(',') if x.strip()]

df_services['service_related'] = df_services['related'].apply(parse_related)

# 4. Ensure case_privacy_related outputs as a boolean (true/false)
def parse_boolean(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip().lower()
    if val_str in ['true', '1', 'yes']:
        return True
    if val_str in ['false', '0', 'no']:
        return False
    return None

df_cases['privacy_related'] = df_cases['privacy_related'].apply(parse_boolean)

# 5. Handle missing data gracefully by converting NaNs to None 
# which will be output as null in JSON instead of NaN
df_services = df_services.replace({np.nan: None})
df_cases = df_cases.replace({np.nan: None})
df_points = df_points.replace({np.nan: None})

# --- PREPARE DATAFRAMES FOR JOINING ---
# Ensure we only keep necessary keys mapped to the desired format

points_rename = {
    'id': 'point_id',
    'title': 'point_title',
    'source': 'point_source',
    'analysis': 'point_analysis',
    'quote_text': 'point_quote_text',
    'quote_start': 'point_quote_start',
    'quote_end': 'point_quote_end',
    'document_id': 'point_document_id',
    'case_id': 'case_id',       # Needed for grouping
    'service_id': 'service_id'  # Needed for grouping
}
available_point_cols = [col for col in points_rename.keys() if col in df_points.columns]
df_points_filtered = df_points[available_point_cols].rename(columns=points_rename)

# Make sure all requested point keys are present
for original, renamed in points_rename.items():
    if renamed not in df_points_filtered.columns and renamed not in ['case_id', 'service_id']:
        df_points_filtered[renamed] = None

cases_rename = {
    'id': 'case_id',
    'classification': 'case_classification',
    'score': 'case_score',
    'title': 'case_title',
    'description': 'case_description',
    'privacy_related': 'case_privacy_related',
    'case_topic': 'case_topic' # Mapped from topic_id
}
available_case_cols = [col for col in cases_rename.keys() if col in df_cases.columns]
df_cases_filtered = df_cases[available_case_cols].rename(columns=cases_rename)

# Ensure all requested case keys are present
for original, renamed in cases_rename.items():
    if renamed not in df_cases_filtered.columns and renamed != 'case_id':
        df_cases_filtered[renamed] = None

# Convert dataframes to dictionaries for optimized grouping and lookups
points_records = df_points_filtered.to_dict('records')
cases_records = df_cases_filtered.to_dict('records')

# Create a lookup dictionary for cases: case_id -> case dictionary
cases_dict = {case['case_id']: case for case in cases_records}

# --- JOIN LOGIC ---
# A Service has many Points (linked via service_id in points.csv)
# A Point maps to one Case (linked via case_id in points.csv)
# Therefore, we first group points by service_id, and within that, group by case_id.

from collections import defaultdict
# Structure: service_id -> case_id -> list of point dictionaries
service_case_points = defaultdict(lambda: defaultdict(list))

for point in points_records:
    s_id = point.get('service_id')
    c_id = point.get('case_id')
    
    # We only process points that are properly linked to a service and a case
    if s_id is not None and c_id is not None:
        # Exclude 'service_id' and 'case_id' from the final point object as per the schema
        p_obj = {k: v for k, v in point.items() if k not in ('service_id', 'case_id')}
        service_case_points[s_id][c_id].append(p_obj)

final_output = []

# Iterate over every single service in services.csv to build the hierarchical JSON array
for _, service in df_services.iterrows():
    s_id = service.get('id')
    
    service_cases_list = []
    
    # If the service has points grouped under cases
    if s_id in service_case_points:
        for c_id, points_list in service_case_points[s_id].items():
            # Look up the details for this case
            case_info = cases_dict.get(c_id, {})
            # Shallow copy to avoid mutating the master dictionary
            case_obj = dict(case_info)
            case_obj['case_id'] = c_id
            
            # Nest the points inside this case
            case_obj['points'] = points_list
            service_cases_list.append(case_obj)
            
    # Construct the final service object matching the exact JSON structure
    service_obj = {
        "service_id": s_id,
        "service_name": service.get('name'),
        "service_url": service.get('url'),
        "cases": service_cases_list,
        "service_related": service.get('service_related', [])
    }
    
    final_output.append(service_obj)

# --- EXPORT TO JSON ---
output_path = '../generated_files/tosdr_services_denormalized.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(final_output, f, indent=4, ensure_ascii=False)

print(f"Data processing complete. Exported {len(final_output)} services to {output_path}")

Data processing complete. Exported 8175 services to ../generated_files/tosdr_services_denormalized.json
